In [1]:
import streamlit as st
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder

# Semilla para reproducibilidad
np.random.seed(42)

# Número de registros
n = 600

# Variables interesantes:
# - brand: marca del auto
# - model_year: año del modelo
# - mileage: kilometraje (en miles)
# - engine_size: tamaño del motor (litros)
# - horsepower: caballos de fuerza
# - fuel_type: tipo de combustible
# - price: precio (variable a predecir)

brands = ['Toyota', 'Honda', 'Ford', 'BMW', 'Audi', 'Hyundai', 'Kia']
fuel_types = ['Gasoline', 'Diesel', 'Hybrid', 'Electric']

# Simulación de datos
data = {
    'brand': np.random.choice(brands, n),
    'model_year': np.random.randint(2005, 2024, n),
    'mileage': np.round(np.random.normal(loc=60, scale=30, size=n), 1),
    'engine_size': np.round(np.random.uniform(1.0, 5.0, n), 1),
    'horsepower': np.random.randint(70, 400, n),
    'fuel_type': np.random.choice(fuel_types, n),
}

# Crear DataFrame
df_autos = pd.DataFrame(data)

# Ajustamos el precio de forma que tenga relación con las variables anteriores
base_price = 5000
brand_modifier = {brand: i * 1000 for i, brand in enumerate(brands)}
fuel_modifier = {'Gasoline': 0, 'Diesel': 1000, 'Hybrid': 3000, 'Electric': 5000}

# Convertimos 'brand' y 'fuel_type' en números
df_autos['brand_modifier'] = df_autos['brand'].map(brand_modifier)
df_autos['fuel_modifier'] = df_autos['fuel_type'].map(fuel_modifier)

# Calculamos el precio basado en las variables
df_autos['price'] = (
    base_price
    + df_autos['brand_modifier']
    + (df_autos['model_year'] - 2005) * 150
    - df_autos['mileage'] * 20
    + df_autos['engine_size'] * 2000
    + df_autos['horsepower'] * 10
    + df_autos['fuel_modifier']
)

# Precio final con algo de ruido
df_autos['price'] = np.round(df_autos['price'] + np.random.normal(0, 2000, n), 2)

df_autos.head()


,brand,model_year,mileage,engine_size,horsepower,fuel_type,brand_modifier,fuel_modifier,price
0,Kia,2023,46.6,2.1,344,Diesel,6000,1000,23188.77
1,BMW,2005,-3.7,3.4,270,Gasoline,3000,0,18724.41
2,Audi,2009,17.0,3.7,353,Gasoline,4000,0,21191.33
3,Kia,2017,104.0,1.9,242,Diesel,6000,1000,18039.55
4,Ford,2008,69.1,2.8,396,Electric,2000,5000,20642.13


In [2]:

# Codificación One-Hot para la marca
df_autos = pd.get_dummies(df_autos, columns=['brand'], drop_first=True)

# Preparar datos para la regresión múltiple
X = df_autos[['model_year', 'mileage', 'engine_size', 'horsepower', 'fuel_modifier'] + [col for col in df_autos.columns if col.startswith('brand_')]] 
y = df_autos['price']  

# Dividir los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Crear el modelo de regresión lineal
model = LinearRegression()

# Entrenar el modelo
model.fit(X_train, y_train)

# Realizar predicciones
y_pred = model.predict(X_test)

# Evaluar el modelo
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Squared Error: {mse}')
print(f'R-squared: {r2}')

Mean Squared Error: 4349086.04697944
R-squared: 0.7936878425408008


In [3]:
import pickle

# Guarda el modelo y las columnas usadas
modelo_info = {
    'model': model,
    'features': X.columns.tolist()
}

with open('modelo_precio_auto.pkl', 'wb') as f:
    pickle.dump(modelo_info, f)
